# Get Index

In [26]:
import re

def get_word_indices(text):
    # This regex finds sequences of non-whitespace characters
    matches = re.finditer(r'\S+', text)
    
    results = []
    for match in matches:
        results.append({
            "word": match.group(),
            "start": match.start(),
            "end": match.end()
        })
    return results

In [40]:
# Example usage:
text_input = "show me s23"
indices = get_word_indices(text_input)

for item in indices:
    print(f"Word: '{item['word']}' | Start: {item['start']} | End: {item['end']}")

Word: 'show' | Start: 0 | End: 4
Word: 'me' | Start: 5 | End: 7
Word: 's23' | Start: 8 | End: 11


# Approach 6

In [36]:
import networkx as nx
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel
import torch
import time


############################################
# CONFIG
############################################

# Threshold per attribute/entity type
THRESHOLD_MAP = {
    "memory": 0.40,
    "color": 0.30,
    "accessory": 0.40,
    "brand": 0.35
}

DEFAULT_THRESHOLD = 0.40


############################################
# LOAD MODELS
############################################

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

tokenizer = AutoTokenizer.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
)
model = AutoModel.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
)


############################################
# SAFE COSINE
############################################

def safe_cosine(a, b):

    a = np.array(a)
    b = np.array(b)

    if np.isnan(a).any() or np.isnan(b).any():
        return 0

    return cosine_similarity([a], [b])[0][0]


############################################
# SPAN EMBEDDINGS
############################################

def generate_span_embeddings(text, entities):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        return_offsets_mapping=True
    )

    offsets = inputs["offset_mapping"][0]
    inputs.pop("offset_mapping")

    with torch.no_grad():
        outputs = model(**inputs)

    token_embeddings = outputs.last_hidden_state[0]

    for ent in entities:

        token_ids = []

        for i, (start, end) in enumerate(offsets):

            if start >= ent["start"] and end <= ent["end"]:
                token_ids.append(i)

        if not token_ids:
            ent["embedding"] = np.zeros(token_embeddings.shape[1])
        else:
            vec = token_embeddings[token_ids].mean(dim=0)
            ent["embedding"] = vec.numpy()

    return entities


############################################
# MINIMUM SPAN DISTANCE
############################################

def span_distance(a, b):

    return max(
        0,
        max(a["start"] - b["end"], b["start"] - a["end"])
    )


############################################
# DISTANCE SCORE
############################################

def distance_score(a, b):

    d = span_distance(a, b)

    return 1 / (d + 1)


############################################
# BUILD GRAPH
############################################

def build_graph(entities):

    G = nx.DiGraph()

    attributes = []
    products = []

    for e in entities:

        G.add_node(
            e["text"],
            label=e["label"],
            start=e["start"],
            end=e["end"]
        )

        if e["label"] == "product_name":
            products.append(e)
        else:
            attributes.append(e)

    for attr in attributes:

        for prod in products:

            sim = safe_cosine(attr["embedding"], prod["embedding"])

            dist_score = distance_score(attr, prod)

            score = 1.0 * sim + 0.0 * dist_score

            G.add_edge(
                attr["text"],
                prod["text"],
                weight=score,
                sim=sim,
                dist_score=dist_score
            )

    return G


############################################
# GET THRESHOLD FOR ATTRIBUTE
############################################

def get_threshold(label):

    return THRESHOLD_MAP.get(label, DEFAULT_THRESHOLD)


############################################
# GRAPH LINKING
############################################

def link_entities(text, entities):

    entities = generate_span_embeddings(text, entities)

    G = build_graph(entities)

    results = []

    # map node → label
    node_labels = {e["text"]: e["label"] for e in entities}

    for node in G.nodes:

        edges = G.out_edges(node, data=True)

        if not edges:
            continue

        best = max(edges, key=lambda x: x[2]["weight"])

        attr_label = node_labels.get(best[0], None)

        threshold = get_threshold(attr_label)

        if best[2]["weight"] >= threshold:

            results.append({
                "attribute": best[0],
                "product": best[1],
                "score": round(best[2]["weight"], 3),
                "threshold": threshold
            })

    return G, results


############################################
# DEBUG GRAPH
############################################

def print_graph(G):

    print("\nGRAPH EDGES\n")

    for u, v, data in G.edges(data=True):

        print(
            f"{u} -> {v} "
            f"| score={data['weight']:.3f} "
            f"| sim={data['sim']:.3f} "
            f"| dist={data['dist_score']:.3f}"
        )

In [37]:
############################################
# TEST
############################################

text = "8gb samsung galaxy s23 vs 32gb chromebook"

entities = [

 {'label': 'memory', 'text': '8gb', 'start': 0, 'end': 3},
 {'label': 'product_name', 'text': 'samsung galaxy s23', 'start': 4, 'end': 22},
 {'label': 'memory', 'text': '32gb', 'start': 26, 'end': 30},
 {'label': 'product_name', 'text': 'chromebook', 'start': 31, 'end': 41}

]

start_time = time.perf_counter()

G, results = link_entities(text, entities)

end_time = time.perf_counter()

duration_ms = (end_time - start_time) * 1000

print(f"Inference Time: {duration_ms:.4f} ms")

print_graph(G)

print("\nFINAL LINKS\n")

for r in results:
    print(r)

Inference Time: 66.2590 ms

GRAPH EDGES

8gb -> samsung galaxy s23 | score=0.599 | sim=0.599 | dist=0.500
8gb -> chromebook | score=0.587 | sim=0.587 | dist=0.034
32gb -> samsung galaxy s23 | score=0.439 | sim=0.439 | dist=0.200
32gb -> chromebook | score=0.439 | sim=0.439 | dist=0.500

FINAL LINKS

{'attribute': '8gb', 'product': 'samsung galaxy s23', 'score': 0.599, 'threshold': 0.4}
{'attribute': '32gb', 'product': 'chromebook', 'score': 0.439, 'threshold': 0.4}


In [34]:
############################################
# TEST
############################################

text = "show me samsung s23 with charger",
entities = [
  {
   "end":19,
   "label":"product_name",
   "start":8,
   "text":"samsung s23"
  },
  {
    "end": 32,
   "label":"accessory",
   "start":25,
   "text":"charger"
  }
 ]

start_time = time.perf_counter()

G, results = link_entities(text, entities)

end_time = time.perf_counter()

duration_ms = (end_time - start_time) * 1000

print(f"Inference Time: {duration_ms:.4f} ms")

print_graph(G)

print("\nFINAL LINKS\n")

for r in results:
    print(r)

Inference Time: 55.5936 ms

GRAPH EDGES

charger -> samsung s23 | score=0.463 | sim=0.463 | dist=0.143

FINAL LINKS

{'attribute': 'charger', 'product': 'samsung s23', 'score': 0.463, 'threshold': 0.4}
